# LangGraph Contract Review Workflow

## 1. Project Overview

**Task:** Build a multi-step **agentic workflow** for reviewing contracts using LangGraph — a framework for building stateful, graph-based LLM applications with explicit control flow, human-in-the-loop checkpoints, and traceable state transitions.

**Why this matters:** Contract review is a perfect case study for LangGraph because it naturally decomposes into sequential steps where:
- Each step produces outputs the next step depends on
- A human must approve edits before finalization (can't fully automate legal decisions)
- State must persist across steps so nothing is lost
- Failures in one step should not silently propagate

**Pipeline Steps:**
1. **Retrieve** — find relevant policy/reference clauses from a knowledge base
2. **Analyze** — identify risks, ambiguities, and missing protections in contract clauses
3. **Suggest Edits** — generate specific redline suggestions with justifications
4. **Human Review** — pause the workflow for human approval/rejection/modification
5. **Final Output** — compile the approved analysis into a structured review document

**Stack:**
- `LangGraph` — state graph orchestration
- `ChatOllama` + `qwen3.5:9b` — LLM (local, no API keys)
- `HuggingFaceEmbeddings` + `ChromaDB` — clause retrieval

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **state graphs**: nodes, edges, conditional edges, and state schemas |
| 2 | Build a **TypedDict state** that flows through all graph nodes |
| 3 | Implement **sequential node chaining** with explicit data dependencies |
| 4 | Add a **human-in-the-loop** checkpoint that pauses execution for review |
| 5 | Use **conditional routing** to branch on human decisions (approve/reject/revise) |
| 6 | Compile and visualize a LangGraph **StateGraph** |
| 7 | Run a full contract review pipeline end-to-end with state inspection |

## 3. What Is a State Graph?

A **state graph** is a directed graph where:
- **Nodes** = functions that read state, do work, and write updated state
- **Edges** = connections that determine which node runs next
- **State** = a shared dictionary (TypedDict) that accumulates results step by step
- **Conditional edges** = edges that route to different nodes based on state values

```
  ┌────────────────────────────────────────────────────────┐
  │                   STATE GRAPH ANATOMY                  │
  ├────────────────────────────────────────────────────────┤
  │                                                        │
  │   START ──→ [Node A] ──→ [Node B] ──→ [Node C]        │
  │                                          │             │
  │                                     conditional        │
  │                                       edge             │
  │                                     ┌───┴───┐          │
  │                                     ▼       ▼          │
  │                                 [Node D] [Node E]      │
  │                                     │       │          │
  │                                     └───┬───┘          │
  │                                         ▼              │
  │                                       END              │
  └────────────────────────────────────────────────────────┘
  
  Key concepts:
  • Each node receives the FULL state and returns a PARTIAL update
  • LangGraph merges the partial update into the state automatically
  • Conditional edges inspect state to decide the next node
  • The graph compiles into an executable runnable
```

### Why State Graphs for Contract Review?

| Alternative | Problem |
|-------------|--------|
| Simple chain (A→B→C) | No branching, no human-in-the-loop |
| Agent with tools | Too autonomous — legal review needs oversight |
| Single mega-prompt | Can't pause, can't inspect intermediate results |
| **State graph** | **Explicit steps, human checkpoints, traceable state** |

## 4. Our Contract Review Graph

```
  START
    │
    ▼
  ┌─────────────┐
  │  RETRIEVE    │  Find relevant reference clauses
  │  REFERENCES  │  from policy knowledge base
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  ANALYZE     │  Identify risks, ambiguities,
  │  CLAUSES     │  and missing protections
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  SUGGEST     │  Generate redline suggestions
  │  EDITS       │  with justifications
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  HUMAN       │  Pause for human review
  │  REVIEW      │ ←── approve / reject / revise
  └──────┬──────┘
         │
    conditional edge
    ┌────┴────┐
    ▼         ▼
  approve   reject/revise
    │         │
    │    ┌────┴────┐
    │    │ RE-ANALYZE│ ──→ back to SUGGEST EDITS
    │    └─────────┘
    ▼
  ┌─────────────┐
  │  COMPILE     │  Produce final structured
  │  OUTPUT      │  review document
  └──────┬──────┘
         │
         ▼
       END
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers langgraph

print("Dependencies: langchain, langgraph, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict, Literal, Annotated
from operator import add

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, START, END
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"LangGraph: StateGraph, START, END imported")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Reference Knowledge Base

## 8. Policy & Best Practice Clauses

Before analyzing a contract, we need a **reference knowledge base** of standard/recommended clauses to compare against. This mimics how a real lawyer references precedent, company policy, or regulatory templates.

In [ ]:
REFERENCE_CLAUSES = [
    {"id": "REF01", "category": "liability",
     "text": "Standard liability cap: Total liability under this agreement shall not exceed the total fees paid by the client in the 12 months preceding the claim. Excludes gross negligence, willful misconduct, and IP indemnification."},

    {"id": "REF02", "category": "liability",
     "text": "Consequential damages exclusion: Neither party shall be liable for indirect, incidental, special, consequential, or punitive damages, including lost profits, lost data, or business interruption, regardless of the theory of liability."},

    {"id": "REF03", "category": "termination",
     "text": "Standard termination clause: Either party may terminate with 30 days written notice. Immediate termination permitted for material breach if not cured within 15 business days of written notice. Data portability obligations survive termination for 90 days."},

    {"id": "REF04", "category": "termination",
     "text": "Termination for convenience: Client may terminate for convenience with 60 days notice. Vendor entitled to payment for work completed through termination date plus reasonable wind-down costs. Prepaid fees refunded on a pro-rata basis."},

    {"id": "REF05", "category": "ip",
     "text": "IP ownership: Client retains all rights to pre-existing IP. Work product created specifically for client is work-for-hire owned by client. Vendor retains rights to pre-existing tools, frameworks, and methodologies. Cross-license granted for embedded vendor IP."},

    {"id": "REF06", "category": "ip",
     "text": "IP indemnification: Vendor shall defend and indemnify client against third-party IP infringement claims arising from the deliverables. Vendor may modify deliverables to avoid infringement or provide a commercially reasonable substitute."},

    {"id": "REF07", "category": "data_privacy",
     "text": "Data processing agreement: Vendor processes personal data only on documented client instructions. Sub-processors require prior written consent. Data breach notification within 48 hours. Annual SOC 2 Type II audit report provided upon request."},

    {"id": "REF08", "category": "data_privacy",
     "text": "Data security minimums: Encryption at rest (AES-256) and in transit (TLS 1.2+). Access controls with principle of least privilege. Regular vulnerability scanning and annual penetration testing. Data retention limited to contract term plus 90 days."},

    {"id": "REF09", "category": "sla",
     "text": "Service level agreement: 99.9% monthly uptime target. Planned maintenance excluded with 48-hour advance notice. Service credits: 10% for 99.0-99.9%, 25% for 95.0-98.9%, 50% for below 95.0%. Credit cap at 30% of monthly fees."},

    {"id": "REF10", "category": "sla",
     "text": "Support response times: Critical issues (P1): 1-hour response, 4-hour resolution target. Major issues (P2): 4-hour response, 8-hour resolution target. Minor issues (P3): 1-business-day response. All times measured during business hours unless 24/7 support is purchased."},

    {"id": "REF11", "category": "payment",
     "text": "Payment terms: Net 30 from invoice date. Late payment interest at 1.5% per month or maximum permitted by law. Disputed invoices must be raised within 15 business days with written justification. Undisputed amounts remain due."},

    {"id": "REF12", "category": "dispute",
     "text": "Dispute resolution: Good-faith negotiation for 30 days. If unresolved, binding arbitration under JAMS rules with a single arbitrator. Arbitration in a mutually agreed location. Each party bears own costs; prevailing party may recover reasonable attorney fees."},
]

print(f"Reference knowledge base: {len(REFERENCE_CLAUSES)} clauses")
print(f"Categories: {sorted(set(c['category'] for c in REFERENCE_CLAUSES))}")

In [ ]:
# Index reference clauses in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("ref_clauses")
except Exception:
    pass

ref_collection = chroma_client.create_collection(
    name="ref_clauses", metadata={"hnsw:space": "cosine"}
)

ref_texts = [c["text"] for c in REFERENCE_CLAUSES]
ref_metas = [{"clause_id": c["id"], "category": c["category"]} for c in REFERENCE_CLAUSES]
ref_ids = [c["id"] for c in REFERENCE_CLAUSES]
ref_vecs = embeddings.embed_documents(ref_texts)
ref_collection.add(documents=ref_texts, embeddings=ref_vecs, metadatas=ref_metas, ids=ref_ids)

print(f"Indexed {ref_collection.count()} reference clauses")

## 9. Contract Under Review

This is a deliberately imperfect vendor contract with risks a reviewer should catch.

In [ ]:
CONTRACT_CLAUSES = [
    {"id": "SEC3.1", "title": "Limitation of Liability",
     "text": "Vendor's total liability under this agreement shall not exceed the fees paid in the most recent calendar month. This limitation applies to all claims including negligence, breach of contract, and indemnification obligations."},

    {"id": "SEC4.2", "title": "Termination",
     "text": "Vendor may terminate this agreement at any time with 7 days written notice. Client may terminate only for cause after providing 90 days written notice and a 60-day cure period."},

    {"id": "SEC5.1", "title": "Intellectual Property",
     "text": "All work product, deliverables, and derivative works created during the engagement shall be the exclusive property of Vendor. Client receives a non-exclusive, revocable license to use deliverables for internal purposes only."},

    {"id": "SEC6.3", "title": "Data Handling",
     "text": "Vendor will use commercially reasonable efforts to protect client data. Vendor may share anonymized data with third-party analytics partners. Data breach notification will be provided within a reasonable timeframe."},

    {"id": "SEC7.1", "title": "Service Levels",
     "text": "Vendor targets 99.5% monthly uptime. No service credits are provided for downtime. Vendor may perform maintenance at any time without advance notice. Support is provided via email on business days."},

    {"id": "SEC8.2", "title": "Payment",
     "text": "All fees are due upon receipt of invoice. Late payments incur interest at 3% per month. Client may not withhold or offset any amounts for any reason, including disputed charges."},
]

print(f"Contract under review: {len(CONTRACT_CLAUSES)} clauses")
for c in CONTRACT_CLAUSES:
    print(f"  [{c['id']}] {c['title']}")

---

# Part B — Define the State Schema

## 10. LangGraph State Design

The **state** is a `TypedDict` that carries all information through the graph. Each node reads what it needs and writes its outputs back.

**Design principle:** Every node should be able to inspect the full history of what happened before it, and downstream nodes should never need to re-derive information.

In [ ]:
class ContractReviewState(TypedDict):
    """State that flows through the contract review graph.

    Each field is populated by a specific node:
      - contract_clauses:   set at START (input)
      - references:         set by retrieve_references node
      - analysis:           set by analyze_clauses node
      - suggested_edits:    set by suggest_edits node
      - human_decision:     set by human_review node
      - human_feedback:     set by human_review node (optional comments)
      - revision_round:     tracks how many revision cycles
      - final_report:       set by compile_output node
      - current_step:       tracks which node last ran
    """
    contract_clauses: list[dict]
    references: list[dict]
    analysis: list[dict]
    suggested_edits: list[dict]
    human_decision: str          # "approve", "reject", "revise"
    human_feedback: str
    revision_round: int
    final_report: str
    current_step: str


print("State schema defined: ContractReviewState")
print("Fields:")
for field_name, field_type in ContractReviewState.__annotations__.items():
    print(f"  {field_name}: {field_type}")

---

# Part C — Build the Graph Nodes

## 11. Node 1: Retrieve References

For each contract clause, retrieve the most relevant reference/policy clauses from our knowledge base. This gives the analyzer context on what "good" looks like.

In [ ]:
def retrieve_references(state: ContractReviewState) -> dict:
    """Node: Retrieve relevant reference clauses for each contract clause."""
    print("  [NODE] retrieve_references")
    references = []

    for clause in state["contract_clauses"]:
        query = f"{clause['title']}: {clause['text']}"
        qv = embeddings.embed_query(query)
        results = ref_collection.query(query_embeddings=[qv], n_results=2)

        refs_for_clause = []
        for i in range(len(results["documents"][0])):
            refs_for_clause.append({
                "ref_id": results["metadatas"][0][i]["clause_id"],
                "category": results["metadatas"][0][i]["category"],
                "text": results["documents"][0][i],
                "similarity": round(1 - results["distances"][0][i], 4),
            })

        references.append({
            "clause_id": clause["id"],
            "clause_title": clause["title"],
            "matched_refs": refs_for_clause,
        })
        print(f"    {clause['id']} → refs: {[r['ref_id'] for r in refs_for_clause]}")

    return {"references": references, "current_step": "retrieve_references"}


print("Node function defined: retrieve_references")

## 12. Node 2: Analyze Clauses

For each contract clause, compare it against the retrieved references and identify **risks**, **ambiguities**, and **missing protections**.

In [ ]:
ANALYZE_SYSTEM = """You are a contract review specialist. Analyze the contract clause by comparing
it to the reference standard clauses. Identify specific risks and issues.
Be precise and cite specific language from both the contract and references."""

ANALYZE_PROMPT = """CONTRACT CLAUSE [{clause_id}] — {clause_title}:
{clause_text}

REFERENCE STANDARDS:
{references}

Analyze this clause. Return JSON:
{{
  "risk_level": "high" or "medium" or "low",
  "risks": ["specific risk 1", "specific risk 2"],
  "ambiguities": ["vague term or missing definition"],
  "missing_protections": ["protection present in reference but absent here"],
  "summary": "one-sentence overall assessment"
}}"""


def analyze_clauses(state: ContractReviewState) -> dict:
    """Node: Analyze each contract clause against reference standards."""
    print("  [NODE] analyze_clauses")
    analysis = []

    for clause, ref_group in zip(state["contract_clauses"], state["references"]):
        ref_text = "\n".join(
            f"[{r['ref_id']}]: {r['text']}" for r in ref_group["matched_refs"]
        )

        prompt = ANALYZE_PROMPT.format(
            clause_id=clause["id"],
            clause_title=clause["title"],
            clause_text=clause["text"],
            references=ref_text,
        )

        result = parse_json(ask(prompt, system=ANALYZE_SYSTEM, temperature=0.1))
        if result:
            result["clause_id"] = clause["id"]
            result["clause_title"] = clause["title"]
        else:
            result = {
                "clause_id": clause["id"],
                "clause_title": clause["title"],
                "risk_level": "unknown",
                "risks": ["Analysis failed — manual review needed"],
                "ambiguities": [],
                "missing_protections": [],
                "summary": "Unable to parse LLM output",
            }

        analysis.append(result)
        print(f"    {clause['id']}: risk={result.get('risk_level', '?')}")

    # Include human feedback context if in a revision round
    feedback = state.get("human_feedback", "")
    if feedback and state.get("revision_round", 0) > 0:
        print(f"    (Incorporating human feedback: {feedback[:60]}...)")

    return {"analysis": analysis, "current_step": "analyze_clauses"}


print("Node function defined: analyze_clauses")

## 13. Node 3: Suggest Edits

Based on the analysis, generate concrete **redline-style** edit suggestions for each clause that has medium or high risk.

In [ ]:
EDIT_SYSTEM = """You are a contract redlining specialist. Generate specific edit suggestions
to fix identified risks. Provide the original problematic language and the
proposed replacement with justification."""

EDIT_PROMPT = """CONTRACT CLAUSE [{clause_id}] — {clause_title}:
{clause_text}

IDENTIFIED ISSUES:
Risk level: {risk_level}
Risks: {risks}
Missing protections: {missing}

{feedback_section}

Suggest edits. Return JSON:
{{
  "edits": [
    {{
      "original": "exact problematic text",
      "proposed": "suggested replacement text",
      "justification": "why this change matters",
      "priority": "critical" or "recommended" or "optional"
    }}
  ]
}}"""


def suggest_edits(state: ContractReviewState) -> dict:
    """Node: Generate edit suggestions for risky clauses."""
    print("  [NODE] suggest_edits")
    all_edits = []

    feedback = state.get("human_feedback", "")
    feedback_section = ""
    if feedback and state.get("revision_round", 0) > 0:
        feedback_section = f"HUMAN FEEDBACK (address this):\n{feedback}"

    for clause, clause_analysis in zip(state["contract_clauses"], state["analysis"]):
        risk = clause_analysis.get("risk_level", "low")
        if risk == "low":
            all_edits.append({
                "clause_id": clause["id"],
                "clause_title": clause["title"],
                "edits": [],
                "note": "Low risk — no edits needed",
            })
            print(f"    {clause['id']}: low risk, skipping")
            continue

        prompt = EDIT_PROMPT.format(
            clause_id=clause["id"],
            clause_title=clause["title"],
            clause_text=clause["text"],
            risk_level=risk,
            risks=clause_analysis.get("risks", []),
            missing=clause_analysis.get("missing_protections", []),
            feedback_section=feedback_section,
        )

        result = parse_json(ask(prompt, system=EDIT_SYSTEM, temperature=0.2))
        edits = result.get("edits", []) if result else []

        all_edits.append({
            "clause_id": clause["id"],
            "clause_title": clause["title"],
            "edits": edits,
        })
        print(f"    {clause['id']}: {len(edits)} edits suggested")

    return {"suggested_edits": all_edits, "current_step": "suggest_edits"}


print("Node function defined: suggest_edits")

## 14. Node 4: Human Review (Simulated)

In a production system, this node would **pause execution** and present the analysis + edits to a human reviewer via a UI. The human decides:
- **approve** → proceed to final output
- **revise** → go back to suggest_edits with feedback
- **reject** → stop the pipeline

Here we simulate this with a pre-configured decision.

In [ ]:
# Simulation configuration — change these to test different paths
SIMULATED_REVIEWS = [
    {
        "decision": "revise",
        "feedback": "The liability cap suggestion is good but also add a requirement "
                     "for mutual liability caps (not just vendor). Also ensure the IP "
                     "clause explicitly addresses open-source components.",
    },
    {
        "decision": "approve",
        "feedback": "Revisions look good. Approve for final compilation.",
    },
]


def human_review(state: ContractReviewState) -> dict:
    """Node: Simulate human review of the analysis and suggested edits.

    In production, this would:
    1. Serialize state to a review interface
    2. Block until the human submits a decision
    3. Resume the graph with the human's input

    LangGraph supports real interrupts via `interrupt_before`/`interrupt_after`
    with a checkpoint database. We simulate here for reproducibility.
    """
    round_num = state.get("revision_round", 0)
    print(f"  [NODE] human_review (round {round_num})")

    # Display what the human sees
    print("\n  ╔══════════════════════════════════════════════════╗")
    print("  ║          HUMAN REVIEW CHECKPOINT                 ║")
    print("  ╠══════════════════════════════════════════════════╣")

    high_risk = [a for a in state["analysis"] if a.get("risk_level") == "high"]
    med_risk = [a for a in state["analysis"] if a.get("risk_level") == "medium"]
    total_edits = sum(len(e.get("edits", [])) for e in state["suggested_edits"])

    print(f"  ║  High-risk clauses:  {len(high_risk):<27}║")
    print(f"  ║  Medium-risk clauses: {len(med_risk):<26}║")
    print(f"  ║  Total edits proposed: {total_edits:<25}║")
    print(f"  ║  Revision round:      {round_num:<26}║")
    print("  ╚══════════════════════════════════════════════════╝")

    # Show edit summary
    for edit_group in state["suggested_edits"]:
        if edit_group.get("edits"):
            print(f"\n    [{edit_group['clause_id']}] {edit_group['clause_title']}:")
            for e in edit_group["edits"]:
                prio = e.get('priority', 'recommended')
                print(f"      [{prio}] {e.get('justification', '')[:65]}")

    # Simulate decision
    review_idx = min(round_num, len(SIMULATED_REVIEWS) - 1)
    sim = SIMULATED_REVIEWS[review_idx]

    print(f"\n  >>> SIMULATED DECISION: {sim['decision'].upper()}")
    print(f"  >>> Feedback: {sim['feedback'][:80]}")

    return {
        "human_decision": sim["decision"],
        "human_feedback": sim["feedback"],
        "revision_round": round_num + 1,
        "current_step": "human_review",
    }


print("Node function defined: human_review")

## 15. Node 5: Compile Final Output

After approval, compile all analysis, approved edits, and references into a structured review document.

In [ ]:
OUTPUT_SYSTEM = """You are a contract review report writer. Compile a clear, structured
final review document from the analysis and approved edits. Use professional legal language.
Organize by clause. Include risk ratings, specific findings, and recommended changes."""

OUTPUT_PROMPT = """Compile a final contract review report from these inputs.

ANALYSIS:
{analysis}

APPROVED EDITS:
{edits}

REVISION ROUNDS: {rounds}
HUMAN FEEDBACK: {feedback}

Write a structured report with:
1. Executive summary (2-3 sentences)
2. Per-clause findings (risk level, issues, recommended changes)
3. Priority action items

Use professional tone. Be specific."""


def compile_output(state: ContractReviewState) -> dict:
    """Node: Compile the final contract review report."""
    print("  [NODE] compile_output")

    analysis_str = json.dumps(state["analysis"], indent=2)[:3000]
    edits_str = json.dumps(state["suggested_edits"], indent=2)[:3000]

    prompt = OUTPUT_PROMPT.format(
        analysis=analysis_str,
        edits=edits_str,
        rounds=state.get("revision_round", 0),
        feedback=state.get("human_feedback", "None"),
    )

    report = ask(prompt, system=OUTPUT_SYSTEM, temperature=0.2)
    print(f"    Report generated: {len(report)} characters")

    return {"final_report": report, "current_step": "compile_output"}


print("Node function defined: compile_output")

---

# Part D — Assemble & Compile the Graph

## 16. Conditional Routing

The **conditional edge** after human review is the key decision point. It inspects `state["human_decision"]` and routes to:
- `"compile_output"` if approved
- `"analyze_clauses"` if revision is requested (loops back through analysis → edits → review)
- `END` if rejected

In [ ]:
def route_after_human_review(state: ContractReviewState) -> str:
    """Conditional edge: decide next node based on human decision."""
    decision = state.get("human_decision", "reject")
    round_num = state.get("revision_round", 0)

    # Safety: limit revision rounds to prevent infinite loops
    if round_num > 3:
        print("    [ROUTE] Max revisions reached — forcing approval")
        return "compile_output"

    if decision == "approve":
        print(f"    [ROUTE] approved → compile_output")
        return "compile_output"
    elif decision == "revise":
        print(f"    [ROUTE] revise → analyze_clauses (round {round_num})")
        return "analyze_clauses"
    else:  # reject
        print(f"    [ROUTE] rejected → END")
        return END


print("Routing function defined: route_after_human_review")
print("  approve → compile_output")
print("  revise  → analyze_clauses (re-enter the analysis loop)")
print("  reject  → END (stop pipeline)")

## 17. Build the State Graph

Now we wire all nodes and edges together into a LangGraph `StateGraph`.

In [ ]:
# Create the graph
workflow = StateGraph(ContractReviewState)

# Add nodes
workflow.add_node("retrieve_references", retrieve_references)
workflow.add_node("analyze_clauses", analyze_clauses)
workflow.add_node("suggest_edits", suggest_edits)
workflow.add_node("human_review", human_review)
workflow.add_node("compile_output", compile_output)

# Add edges (sequential flow)
workflow.add_edge(START, "retrieve_references")
workflow.add_edge("retrieve_references", "analyze_clauses")
workflow.add_edge("analyze_clauses", "suggest_edits")
workflow.add_edge("suggest_edits", "human_review")

# Conditional edge after human review — the branching point
workflow.add_conditional_edges(
    "human_review",
    route_after_human_review,
    {
        "compile_output": "compile_output",
        "analyze_clauses": "analyze_clauses",
        END: END,
    },
)

# Final edge
workflow.add_edge("compile_output", END)

# Compile
app = workflow.compile()

print("Graph compiled successfully!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print("\nGraph structure:")
print("  START → retrieve_references → analyze_clauses → suggest_edits → human_review")
print("  human_review ──(approve)──→ compile_output → END")
print("  human_review ──(revise)───→ analyze_clauses (loop)")
print("  human_review ──(reject)───→ END")

## 18. Visualize the Graph

LangGraph can render the graph as a Mermaid diagram. Let's show the text representation.

In [ ]:
# Print the graph structure as Mermaid (text representation)
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nManual graph description:")
    print("  __start__ --> retrieve_references")
    print("  retrieve_references --> analyze_clauses")
    print("  analyze_clauses --> suggest_edits")
    print("  suggest_edits --> human_review")
    print("  human_review --(approve)--> compile_output")
    print("  human_review --(revise)--> analyze_clauses")
    print("  human_review --(reject)--> __end__")
    print("  compile_output --> __end__")

---

# Part E — Execute the Workflow

## 19. Run the Full Contract Review

In [ ]:
# Prepare initial state
initial_state = {
    "contract_clauses": CONTRACT_CLAUSES,
    "references": [],
    "analysis": [],
    "suggested_edits": [],
    "human_decision": "",
    "human_feedback": "",
    "revision_round": 0,
    "final_report": "",
    "current_step": "start",
}

print("EXECUTING CONTRACT REVIEW WORKFLOW")
print("=" * 60)
print(f"Contract: {len(CONTRACT_CLAUSES)} clauses")
print(f"Reference KB: {ref_collection.count()} standard clauses")
print(f"Simulated reviews: {len(SIMULATED_REVIEWS)} rounds")
print("=" * 60)
print()

In [ ]:
# Execute with step-by-step streaming
print("Running graph step by step...\n")

step_count = 0
final_state = None

for step_output in app.stream(initial_state, {"recursion_limit": 25}):
    step_count += 1
    for node_name, node_output in step_output.items():
        current = node_output.get("current_step", node_name)
        print(f"\n--- Step {step_count}: {node_name} completed ---")

        # Show key state changes
        if "references" in node_output:
            print(f"    → Retrieved references for {len(node_output['references'])} clauses")
        if "analysis" in node_output:
            risks = [a.get('risk_level', '?') for a in node_output['analysis']]
            print(f"    → Analysis complete: {dict((k, risks.count(k)) for k in set(risks))}")
        if "suggested_edits" in node_output:
            edits_count = sum(len(e.get('edits', [])) for e in node_output['suggested_edits'])
            print(f"    → {edits_count} edits suggested")
        if "human_decision" in node_output:
            print(f"    → Human decision: {node_output['human_decision']}")
        if "final_report" in node_output:
            print(f"    → Final report: {len(node_output['final_report'])} chars")

    final_state = step_output

print(f"\nWorkflow complete in {step_count} steps.")

## 20. Inspect Final State

In [ ]:
# Get the final state from the last step output
# Run the graph again with invoke to get complete final state
result_state = app.invoke(initial_state, {"recursion_limit": 25})

print("FINAL STATE INSPECTION")
print("=" * 60)

print(f"\nCurrent step:    {result_state.get('current_step', '?')}")
print(f"Revision rounds: {result_state.get('revision_round', 0)}")
print(f"Human decision:  {result_state.get('human_decision', '?')}")

print(f"\nReferences retrieved: {len(result_state.get('references', []))} clause groups")
print(f"Analysis results:    {len(result_state.get('analysis', []))} clauses analyzed")
print(f"Edit suggestions:    {sum(len(e.get('edits', [])) for e in result_state.get('suggested_edits', []))} total edits")
print(f"Final report:        {len(result_state.get('final_report', ''))} characters")

---

# Part F — Review Results in Detail

## 21. Analysis Results

In [ ]:
print("CLAUSE ANALYSIS RESULTS")
print("=" * 90)

for a in result_state["analysis"]:
    risk = a.get("risk_level", "?")
    icon = {"high": "!!!", "medium": " ! ", "low": " . "}.get(risk, " ? ")
    print(f"\n  [{icon}] {a['clause_id']} — {a.get('clause_title', '')} [{risk.upper()}]")
    print(f"      Summary: {a.get('summary', 'N/A')[:80]}")

    if a.get("risks"):
        print(f"      Risks:")
        for r in a["risks"][:3]:
            print(f"        - {r[:75]}")

    if a.get("missing_protections"):
        print(f"      Missing protections:")
        for m in a["missing_protections"][:3]:
            print(f"        - {m[:75]}")

    if a.get("ambiguities"):
        print(f"      Ambiguities:")
        for am in a["ambiguities"][:2]:
            print(f"        - {am[:75]}")

## 22. Suggested Edits

In [ ]:
print("SUGGESTED EDITS (REDLINES)")
print("=" * 90)

for edit_group in result_state["suggested_edits"]:
    edits = edit_group.get("edits", [])
    if not edits:
        note = edit_group.get("note", "No edits")
        print(f"\n  [{edit_group['clause_id']}] {edit_group.get('clause_title', '')}")
        print(f"    {note}")
        continue

    print(f"\n  [{edit_group['clause_id']}] {edit_group.get('clause_title', '')} — {len(edits)} edits")

    for i, e in enumerate(edits, 1):
        prio = e.get("priority", "recommended").upper()
        print(f"\n    Edit {i} [{prio}]:")
        orig = e.get("original", "")
        if orig:
            print(f"      ORIGINAL:  {orig[:80]}")
        proposed = e.get("proposed", "")
        if proposed:
            print(f"      PROPOSED:  {proposed[:80]}")
        just = e.get("justification", "")
        if just:
            print(f"      REASON:    {just[:80]}")

## 23. Final Report

In [ ]:
print("FINAL CONTRACT REVIEW REPORT")
print("=" * 90)

report = result_state.get("final_report", "")
if report:
    wrap_print(report)
else:
    print("  No final report generated (possibly rejected during human review).")

---

# Part G — Understanding the Graph Execution

## 24. State Flow Visualization

Understanding **how state flows** through the graph is key to debugging and extending workflows.

In [ ]:
print("STATE FLOW THROUGH THE GRAPH")
print("=" * 70)
print()
print("  ┌─ START ────────────────────────────────────────────────┐")
print("  │  State: contract_clauses=[6 clauses], everything else empty  │")
print("  └──────────────────────────┬─────────────────────────────┘")
print("                             │")
print("  ┌─ retrieve_references ────┴─────────────────────────────┐")
print("  │  READS:   contract_clauses                             │")
print("  │  WRITES:  references (12 matched refs across 6 clauses)│")
print("  │  HOW:     embeds each clause → vector search → top-2   │")
print("  └──────────────────────────┬─────────────────────────────┘")
print("                             │")
print("  ┌─ analyze_clauses ────────┴─────────────────────────────┐")
print("  │  READS:   contract_clauses, references                 │")
print("  │  WRITES:  analysis (risk_level, risks, ambiguities)    │")
print("  │  HOW:     LLM compares clause vs reference standards   │")
print("  └──────────────────────────┬─────────────────────────────┘")
print("                             │")
print("  ┌─ suggest_edits ──────────┴─────────────────────────────┐")
print("  │  READS:   contract_clauses, analysis, human_feedback   │")
print("  │  WRITES:  suggested_edits (original→proposed pairs)    │")
print("  │  HOW:     LLM generates redlines for medium/high risk  │")
print("  └──────────────────────────┬─────────────────────────────┘")
print("                             │")
print("  ┌─ human_review ───────────┴─────────────────────────────┐")
print("  │  READS:   analysis, suggested_edits                    │")
print("  │  WRITES:  human_decision, human_feedback, revision_round│")
print("  │  HOW:     displays summary, simulates approve/revise   │")
print("  └──────────────────────────┬─────────────────────────────┘")
print("                        conditional")
print("                     ┌───────┴───────┐")
print("                 approve         revise → back to analyze")
print("                     │")
print("  ┌─ compile_output ─┴─────────────────────────────────────┐")
print("  │  READS:   analysis, suggested_edits, human_feedback    │")
print("  │  WRITES:  final_report                                 │")
print("  │  HOW:     LLM compiles structured review document      │")
print("  └──────────────────────────┬─────────────────────────────┘")
print("                             │")
print("                           END")

## 25. Testing the Reject Path

Let's verify the graph handles rejection correctly by running with a different simulated decision.

In [ ]:
# Test the reject path
original_reviews = SIMULATED_REVIEWS.copy()
SIMULATED_REVIEWS.clear()
SIMULATED_REVIEWS.append({
    "decision": "reject",
    "feedback": "Contract terms are fundamentally unacceptable. Do not proceed.",
})

reject_state = {
    "contract_clauses": CONTRACT_CLAUSES[:2],  # Just first 2 clauses for speed
    "references": [], "analysis": [], "suggested_edits": [],
    "human_decision": "", "human_feedback": "",
    "revision_round": 0, "final_report": "", "current_step": "start",
}

print("TESTING REJECT PATH")
print("=" * 50)

reject_result = app.invoke(reject_state, {"recursion_limit": 10})

print(f"\nFinal decision: {reject_result.get('human_decision')}")
print(f"Final report:   {'(empty — rejected)' if not reject_result.get('final_report') else 'Generated'}")
print(f"Current step:   {reject_result.get('current_step')}")
print("\nWhen rejected, the graph stops at human_review → END.")
print("No final report is compiled. This is correct behavior.")

# Restore original reviews
SIMULATED_REVIEWS.clear()
SIMULATED_REVIEWS.extend(original_reviews)

## 26. Evaluation — How Well Did the Review Work?

In [ ]:
# Check what the review should have caught
EXPECTED_ISSUES = {
    "SEC3.1": {
        "risk_level": "high",
        "key_issues": [
            "Liability capped at one month's fees (too low — standard is 12 months)",
            "Cap applies to all claims including negligence/indemnification (no carve-outs)",
        ],
    },
    "SEC4.2": {
        "risk_level": "high",
        "key_issues": [
            "Asymmetric termination: vendor gets 7 days, client needs 90 days + 60-day cure",
            "No termination for convenience for client",
        ],
    },
    "SEC5.1": {
        "risk_level": "high",
        "key_issues": [
            "Vendor owns all work product (client should own custom deliverables)",
            "Client license is revocable (should be perpetual/irrevocable)",
        ],
    },
    "SEC6.3": {
        "risk_level": "high",
        "key_issues": [
            "Vague 'commercially reasonable efforts' for data security",
            "Data sharing with third parties without consent",
            "No specific breach notification timeline",
        ],
    },
    "SEC7.1": {
        "risk_level": "medium",
        "key_issues": [
            "No service credits for downtime",
            "Unscheduled maintenance without notice",
        ],
    },
    "SEC8.2": {
        "risk_level": "medium",
        "key_issues": [
            "Payment due on receipt (no Net-30)",
            "3% monthly interest is very high",
            "No right to dispute charges",
        ],
    },
}

print("EVALUATION: Did the pipeline catch the expected issues?")
print("=" * 80)

for a in result_state["analysis"]:
    cid = a["clause_id"]
    expected = EXPECTED_ISSUES.get(cid, {})
    exp_risk = expected.get("risk_level", "?")
    got_risk = a.get("risk_level", "?")

    risk_match = got_risk == exp_risk or (got_risk in ("high", "medium") and exp_risk in ("high", "medium"))
    icon = "+" if risk_match else "-"

    print(f"\n  [{icon}] {cid}: expected={exp_risk}, got={got_risk}")

    # Check if key issues were identified (fuzzy match)
    all_findings = " ".join(str(v) for v in [
        a.get("risks", []), a.get("ambiguities", []),
        a.get("missing_protections", []), a.get("summary", ""),
    ]).lower()

    for issue in expected.get("key_issues", []):
        # Check for key words from the expected issue
        words = [w for w in issue.lower().split() if len(w) > 4]
        hits = sum(1 for w in words if w in all_findings)
        found = hits >= len(words) * 0.4  # at least 40% of key words
        mark = "+" if found else "-"
        print(f"    [{mark}] {issue[:70]}")

print("\n  Key: [+] found/matched, [-] missed")

---

## 27. Common Mistakes with LangGraph

| Mistake | Why It's Wrong | Fix |
|---------|---------------|-----|
| Returning full state from nodes | LangGraph merges partial updates — returning everything overwrites | Return only the fields you changed |
| No recursion limit | Revision loops can run forever | Set `recursion_limit` in config |
| Skipping state validation | Nodes receive invalid data and fail silently | Add type checks at node entry |
| Putting all logic in one mega-node | Loses the benefit of step-by-step inspection | Split into focused, single-responsibility nodes |
| Not testing all conditional paths | Reject/revise paths may be broken without testing | Run each branch with test data |
| Ignoring state schema | Nodes write fields that downstream nodes don't expect | Define a clear TypedDict and stick to it |

## 28. Production Considerations

1. **Real human-in-the-loop**: Use `interrupt_before=["human_review"]` with a checkpoint database (SQLite/Postgres) to truly pause and resume the graph
2. **Persistence**: Save state snapshots at each step for audit trails
3. **Error handling**: Wrap node functions with retry logic and fallback outputs
4. **Parallel nodes**: If clauses are independent, analyze them in parallel using LangGraph's parallel fan-out
5. **Streaming**: Stream LLM outputs within nodes for better UX
6. **Version control**: Record which graph version and LLM model produced each review

## 29. Exercises

### Exercise 1
Add a **risk summary node** between `analyze_clauses` and `suggest_edits` that produces a one-page risk heat map. Only clauses above a configurable risk threshold should proceed to edit suggestions.

### Exercise 2
Implement **real human input**: replace the simulated review with `input()` calls so the user can interactively approve, reject, or add feedback. Handle invalid input gracefully.

### Exercise 3
Add a **parallel analysis** node that fans out across contract clauses simultaneously using LangGraph's `Send` API, then collects results. Compare latency vs the sequential approach.

### Mini Challenge
Build a **contract comparison** workflow: take two versions of a contract (original and redlined), align clauses between them, and produce a change analysis showing what improved, what regressed, and what's new. Use a graph with `diff` and `evaluate_changes` nodes.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Contract reviewed:     {len(CONTRACT_CLAUSES)} clauses")
print(f"Reference KB:          {len(REFERENCE_CLAUSES)} standard clauses")
print(f"Revision rounds:       {result_state.get('revision_round', 0)}")
print(f"Final decision:        {result_state.get('human_decision', '?')}")
print(f"Final report length:   {len(result_state.get('final_report', ''))} chars")
print()
risk_counts = {}
for a in result_state.get("analysis", []):
    r = a.get("risk_level", "unknown")
    risk_counts[r] = risk_counts.get(r, 0) + 1
print(f"Risk distribution:     {risk_counts}")
total_edits = sum(len(e.get('edits', [])) for e in result_state.get('suggested_edits', []))
print(f"Total edits proposed:  {total_edits}")
print()
print(f"Graph components built:")
print(f"  1. Reference retrieval node    — vector search for relevant standards")
print(f"  2. Clause analysis node        — LLM risk assessment vs references")
print(f"  3. Edit suggestion node        — redline generation with justifications")
print(f"  4. Human review node           — checkpoint with approve/revise/reject")
print(f"  5. Conditional routing          — branch on human decision")
print(f"  6. Final compilation node      — structured review document")
print(f"  7. State schema                — TypedDict with full traceability")
print(f"  8. Reject path tested          — verified early termination works")

## 30. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **State graphs** decompose complex workflows into inspectable, testable steps with explicit data flow |
| 2 | **TypedDict state** ensures every node knows what data is available and what it should produce |
| 3 | **Nodes return partial updates** — LangGraph merges them into the state automatically |
| 4 | **Conditional edges** let you implement decision points (approve/reject/revise) cleanly |
| 5 | **Human-in-the-loop** is a first-class concept — pause, review, and resume with feedback |
| 6 | **Revision loops** with a recursion limit prevent infinite cycles while allowing iterative improvement |
| 7 | **Reference retrieval + LLM analysis** is more reliable than LLM-only review because the model has concrete standards to compare against |
| 8 | **Every graph path must be tested** — approve, reject, and revise paths can each have different bugs |
| 9 | **State inspection** at any step makes debugging straightforward — you can see exactly what each node produced |
| 10 | **Contract review is a natural fit** for state graphs because it requires sequential reasoning, human oversight, and iterative refinement |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*